# 00 · Environment, identity, and the credit meter

## Goal

Stand up the one thing every later notebook depends on: a Dataverse
environment, two working identities (delegated for local runs, service
principal for CI), a DLP posture you've actually looked at, and a credit
budget you set *before* anything bills against it.

By the end of this notebook you have: an environment, `pac auth` profiles
for both identity paths, a recorded credit budget, and a single
prerequisite report listing every tenant/admin switch the rest of the
curriculum needs — so you raise one ticket, not nine.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
import sys, subprocess
assert sys.version_info >= (3, 10), "Use Python 3.10+"

result = subprocess.run(["pac", "--version"], capture_output=True, text=True)
assert result.returncode == 0, "pac CLI not on PATH — install Power Platform CLI first"
print(result.stdout.strip())
# Pin this. cli-copilot authoring mode (used from notebook 01 onward) is
# newer than classic authoring and its YAML schema is expected to move —
# record the exact version you validated this curriculum against.
PAC_VERSION_PINNED = result.stdout.strip()


If that assertion failed: install the CLI, then re-run this cell. Nothing below works without it.


## Concept

**Why this notebook front-loads so much administration:** finding #2 is the
one that bites POC teams hardest — the GitHub Copilot harness (the one this
curriculum uses throughout, see `01`) bills Copilot Credits from the moment
you *build*. Previewing, testing, and evaluating an agent all meter, and an
M365 Copilot licence does not cover it. If you don't set a budget and watch
it before notebook `01` publishes anything, notebook `22`'s multi-agent
fan-out will be the first time anyone notices the invoice.

**Why a prerequisite report, not nine separate surprises:** Tracks 5
(Fabric/Work/Web IQ), 6 (multi-agent model tiering), and 7 (observability)
each depend on tenant-level admin switches your POC team may not control —
four independent model-availability toggles alone (finding #5). Discovering
these one at a time, as a cryptic 403 three notebooks deep, wastes a review
cycle each time. `csx.admin.prerequisite_report()` exists so you find out
now, in one pass, what to ask your tenant admin for.


## Build


### Identity: delegated (local) and application (CI)

Both need `CopilotStudio.Copilots.Invoke` — delegated on the signed-in user, application on the service principal. Notebook `01` exercises both so the split is felt once, early.


In [ ]:
from csx.pac import auth_create, version

print(version())

# Delegated profile — interactive, used from notebooks run locally.
auth_create(
    environment_url="https://<your-env>.crm.dynamics.com",  # from the environment you create/checkpoint below
    name="crd-delegated",
)

# Application profile — service principal, used by CI (infra/pipelines/deploy.yml)
# and by any notebook run with delegated=False in csx.clients.get_copilot_client.
import os
auth_create(
    environment_url="https://<your-env>.crm.dynamics.com",
    name="crd-application",
    application_id=os.environ.get("APP_CLIENT_ID"),
    client_secret=os.environ.get("APP_CLIENT_SECRET"),
    tenant_id=os.environ.get("TENANT_ID"),
)


### Checkpoint: the Dataverse environment actually exists

Create it via the Power Platform admin center, `pac admin create-environment`, or — the reproducible option — `T0-bonus`'s Terraform. Whichever you use, this cell is what actually verifies it, rather than trusting that the click worked.


In [ ]:
from csx.checkpoint import checkpoint
import subprocess, json

def probe_environment():
    result = subprocess.run(["pac", "admin", "list", "--json"], capture_output=True, text=True)
    envs = json.loads(result.stdout) if result.returncode == 0 else []
    return next((e for e in envs if "contract-renewal-desk" in e.get("DisplayName", "").lower()), None)

env = checkpoint(
    name="Dataverse environment exists",
    probe=probe_environment,
    remediation=(
        "Create an environment named to include 'Contract Renewal Desk' — "
        "via PPAC, `pac admin create-environment`, or `terraform apply` in "
        "infra/terraform/platform (T0-bonus)."
    ),
)
print(env)


### DLP posture — read it, don't assume it

You're not writing a policy yet (that's `T8-bonus` / notebook `24`); you're finding out what's already blocked so knowledge/tool notebooks don't fail mysteriously.


In [ ]:
import subprocess, json

result = subprocess.run(["pac", "admin", "list-dlp-policy", "--json"], capture_output=True, text=True)
policies = json.loads(result.stdout) if result.returncode == 0 else []
for p in policies:
    print(p.get("displayName"), "->", p.get("environmentType"))


### Credit budget — set before anything bills

Pick a number your admin actually agreed to, put it in `.env`, and every notebook's cost cell from here on checks against it.


In [ ]:
budget = 5000  # credits — replace with the number you actually agreed with finance/admin
with open("../.env", "a") as f:
    f.write(f"\nCOPILOT_CREDIT_BUDGET={budget}\n")
print(f"Credit budget set to {budget}. csx.cost.CreditMeter.report_cost() will warn at 90% of this.")


### The prerequisite report


In [ ]:
from csx.admin import prerequisite_report

# Fill this in by actually checking each surface — PPAC environment/group
# settings, M365 admin center per-provider approval, Fabric admin portal.
# Leaving a value as None is honest; guessing True is not.
known_status = {
    "external_models_ppac": None,
    "external_models_provider": None,
    "preview_experimental_models": None,
    "move_data_across_regions": None,
    "fabric_cross_geo": None,
    "otel_span_export": None,
}
print(prerequisite_report(known_status))


## Verify

Same harness, same golden set, every notebook.


Nothing to run against a published agent yet — that starts in `01`. This notebook's verification is the checkpoint above plus the printed report: read it before continuing.


## Cost


In [ ]:
from csx.cost import CreditMeter
meter = CreditMeter(environment_id="pending")  # environment_id filled in once 01 publishes
meter.report_cost("00", budget=5000, delta_credits=0, note="baseline — environment creation and auth setup do not meter")


## Teardown


In [ ]:
# Nothing to tear down yet — the environment and identities are the
# foundation the rest of the curriculum builds on. T0-bonus shows the full
# `terraform destroy` drill for when the whole workshop environment is done.
print("No teardown for 00 — this environment is reused through notebook 25.")
